# Python循环

## for 循环
- 由可迭代对象提供“下一个元素”（不要求可迭代对象的有序性）
- 正常情况下在元素耗尽时终止

### 逐项处理数据
- for 从可迭代对象取得元素
    - 可迭代不等于必须能用索引
    - for 需要的是迭代协议，不要求对象一定支持 items[0]
- 每得到一个元素，就把循环变量重新绑定到该元素
- 然后执行一次循环体
- 正常情况下在元素耗尽时终止

In [ ]:
metadata_records = [
    {"field": "customer_id", "nullable": False},
    {"field": "phone", "nullable": True},
    {"field": "created_at", "nullable": False},
]

for record in metadata_records:
    print(record["field"])

In [ ]:
print(record)

- 循环结束后，record 在普通函数或模块作用域中通常仍绑定最后一个元素
- 循环不会自动建立独立作用域,因此为了避免循环变量覆盖掉同作用域中的重要变量
    - 优先使用专用、含义明确的循环变量名
    - 也可以手动删除或者重置

In [ ]:
record = None # 重置

del record # 删除

print(record)

### 在循环中累积结果
- 循环反复读取输入记录，并将满足条件的结果写入循环外部创建的容器
- 结果列表必须在循环开始前创建；每轮只负责判断当前元素，然后选择是否把结果追加进去

In [ ]:
metadata_records = [
    {"field": "customer_id", "nullable": False},
    {"field": "phone", "nullable": True},
    {"field": "created_at", "nullable": False},
]

nullable_fields = []

for record in metadata_records:
    if record["nullable"]:
        nullable_fields.append(record["field"])

print(nullable_fields)

- 循环本身只负责重复执行
- 跨轮次保存结果依赖循环外部的可变状态
- 如果把需要累积的容器写在循环里面，每轮都会重新创建空容器，前面轮次的结果会丢失
- 即累积容器通常在循环外创建，在循环内更新

## while 循环

### 按状态重复
- 每轮开始前重新计算条件
    - 条件为 True 执行循环体，执行完后回到入口重新计算
    - 条件为 False 立即结束
- while 不知道你最终想做几次，能否结束完全依赖条件以及循环体是否让条件逐渐接近 False（关键在于状态的正确更新）
- 如果循环体不改变影响条件的状态，循环可能永远不会停止

In [ ]:
pending_issues = [
    "主键重复",
    "手机号未脱敏",
    "日期格式错误",
]

processed_count = 0

while pending_issues:
    current_issue = pending_issues.pop(0) # 每次循环取出第一个元素
    print("处理：", current_issue)
    processed_count += 1
# 随着所有元素被取出，pending_issues变成falsy值（空列表）

print("处理数量：", processed_count)

## continue、break

### continue：跳过当前循环
- continue 立即结束当前这一轮的循环体，不执行它后面的语句，然后返回循环入口：
    - 遇到 continue
    - 当前轮剩余语句全部跳过
    - 返回循环入口
        - for：请求下一个元素
        - while：重新计算条件

In [ ]:
metadata_records = [
    {"field": "customer_id", "active": True},
    {"field": "legacy_code", "active": False},
    {"field": "phone", "active": True},
]

for record in metadata_records:
    if not record["active"]:
        continue

    print("执行治理检查：", record["field"])

- 放在 while 中时，必须确保必要的状态更新不会被一起跳过，否则可能形成无限循环

In [ ]:
index = 0

while index < 3:
    if index == 1:
        index += 1
        continue

    index += 1

### break：终止整个循环
- break 立即停止最近一层循环，控制流直接移动到该循环之后：
    - 遇到 break
    - 不再执行当前轮剩余代码，不再进入后续轮次
    - 跳到循环之后

In [ ]:
quality_issues = [
    {"rule_id": "DQ-01", "severity": "LOW"},
    {"rule_id": "DQ-02", "severity": "CRITICAL"},
    {"rule_id": "DQ-03", "severity": "MEDIUM"},
]
count =0
for issue in quality_issues:
    print("检查：", issue["rule_id"])
    count += 1

    if issue["severity"] == "CRITICAL":
        print("发现严重问题，停止后续发布检查")
        break
    if count == len(quality_issues):
        print("已完成全部检查，没有发现严重问题")
        
print("程序结束")

检查： DQ-01
检查： DQ-02
发现严重问题，停止后续发布检查
程序结束


- break 只能出现在循环内部，终止的是最近一层循环
- break 不会自动退出函数，也不会自动退出所有嵌套循环

## 嵌套循环
- 外层循环每取得一个元素
- 内层循环都会针对该外层元素执行自己的完整遍历

In [7]:
tables = [
    {
        "table": "customer",
        "fields": ["customer_id", "phone"],
    },
    {
        "table": "orders",
        "fields": ["order_id", "amount"],
    },
]

for table in tables:
    for field in table["fields"]:
        print(table["table"], "->", field)

customer -> customer_id
customer -> phone
orders -> order_id
orders -> amount


- 内外层循环都必须是完整的循环，有自己的可迭代对象和循环变量
- 内层循环基于外层循环取得的元素，不能独立启动
- 当循环嵌套过多时，算法复杂度会快速增加

In [11]:
tables = [
    {
        "table": "customer",
        "fields": ["customer_id", "phone"],
    },
    {
        "table": "orders",
        "fields": ["order_id", "amount"],
    },
]

for table in tables:
    for field in table["fields"]:
        if field == "customer_id":
            print("发现敏感字段：", table["table"], field)
            break
        print(f'完成扫描{table["table"]}->{field}')

发现敏感字段： customer customer_id
完成扫描orders->order_id
完成扫描orders->amount


- break 退出了 fields 内层循环，但没有退出 tables 外层循环
- break 只终止包含它的最近一层循环，也可以理解为终止基于当前外层循环所提供元素的内层循环
- 如果 break 位于内层循环，外层循环仍会继续取得下一个元素，然后继续基于这个元素的内层循环


## else（break配套）
- 当循环体内存在break时，通过设置else代码体，处理未通过break终止时的情况
- 通过设置else，避免了手动设置标志变量和逻辑无法收拢的情况

In [ ]:
quality_issues = [
    {"rule_id": "DQ-01", "severity": "LOW"},
    {"rule_id": "DQ-02", "severity": "CRITICAL"},
]

for issue in quality_issues:
    if issue["severity"] == "CRITICAL":
        print("发现严重问题：", issue["rule_id"])
        break
else:
    print("未发现严重问题")